In [0]:
data =[ 
    (101,'Gopal','IT',450000),
    (102,'Roshan','HR',55000),
    (103,'Nitin','Finance',65000),
    (104,'Dipak','Insurance',76000),
    (105,'Lalit','Production',56000)
]

columns = ['id','name','department','salary']
df = spark.createDataFrame(data,columns)
df.show()

Filter employees whose salary is greater than 50000.



Display only the name, department, and salary columns.



Store the final result in a variable named result.



Display the result.

In [0]:
from pyspark.sql.functions import *
df_fillterd = df.filter(col("salary")> 60000)
df_result = df_fillterd.select("name","department","salary")

result = df_result
result.show()


Add a new column called incremented_salary.



Employees in the IT department receive a 10% increment.



All other employees receive a 5% increment.



Display name, department, salary, and incremented_salary.



Store the final DataFrame in result.

In [0]:
from pyspark.sql.functions import col , when
df1 = df.withColumn(
    "increment_salary",
when(
     col("department")=="IT",
     col("salary")*1.10
)    .otherwise(
        col("salary")*1.05
)
)
result = df1.select("name","department","salary","increment_salary")
result.explain("extended")
result.show()


Group employees by department.



Calculate the average salary.



Count the employees in each department.



Name the output columns avg_salary and employee_count.



Store the final DataFrame in result and display it.

In [0]:
from pyspark.sql.functions import col , lit , avg ,count
result = df.groupBy("department").agg(
    avg(col("salary")).alias("avrage_salary"),
    count("*").alias('employee_count')
    )
           

result.show()




In [0]:
data = [
    (1, "Pune", 2500),
    (2, "Mumbai", 4000),
    (3, "Pune", 3500),
    (4, "Nashik", 1500),
    (5, "Mumbai", 2000),
    (6, "Pune", 1800),
    (7, "Nashik", 4500),
    (8, "Mumbai", 1200)
]

columns = ["order_id", "city", "amount"]

df = spark.createDataFrame(data, columns)

df.show()

Keep only orders where amount > 1000.



Group the remaining orders by city.



Calculate total sales per city as total_sales.



Keep only cities where total_sales > 5000.



Store the result in result and display it.

In [0]:
df1 = df.filter(col("amount")>1000)
df2 = df1.groupBy("city").agg(
    sum("amount").alias("total_sales")
)
df_result =df2.filter(col("total_sales")>5000
)
result = df_result
result.explain("extended")
result.show()








Group by city.



Calculate total_sales, order_count, and average_order_amount.



Keep cities with at least 2 orders and total sales greater than 5000.



Sort by total_sales in descending order.



Store the final result in result.

In [0]:

from pyspark.sql.functions import *
df1 = df.groupBy("city").agg(
    sum('amount').alias("total_sales"),
    avg("amount").alias("avrage_amount"),
    count("*").alias('total_order')
)
df2 = df1.filter(
(col("total_order")>=2)&
(col("total_sales")>5000)
)

result = df2.orderBy(
    col("total_sales").desc()
)
result.show()



In [0]:
data = [

    (1,"C101",2500,"2026-01-10"),
    (2,"C102",4000,"2026-01-12"),
    (3,"C103",3500,"2026-02-05"),
    (4, "C103", 1500, "2026-01-15"),
    (5, "C102", 2000, "2026-03-01"),
    (6, "C101", 1800, "2026-03-10"),
    (7, "C103", 4500, "2026-02-20"),
    (8, "C102", 1200, "2026-03-15")
]
columns = ["order_id","customer_id","amount","order_date"]
df = spark.createDataFrame(data,columns)
df.show()

Convert order_date to a date type.



Partition records by customer_id.



Order each customer's records by order_date descending.



Assign a row number to each order.



Keep only the latest order for each customer.



Display customer_id, order_id, amount, and order_date.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df1 = df.withColumn(
    "order_date",
    to_date(col("order_date"))
)
window_spece = Window.partitionBy(
    "customer_id",
).orderBy(
    col('order_date').desc()

)
df2 = df1.withColumn(
    "row_num",
    row_number().over(window_spece)
    
)
df3 = df2.filter(
    col("row_num")==1
)
result = df3.select("customer_id","amount","order_date")
result.show()


Create a window partitioned by department.



Rank salaries from highest to lowest using dense_rank().



Filter the results to keep only rank 2.



Display emp_id, name, department, and salary.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
window_Spect = Window.partitionBy(
    'department'
).orderBy(
    col('amount').desc()
) 
df2 = df.withColumn(
    "salary_rank",
    dense_rank().over(window_Spect)
)

result = df2.filter(
    col("salary_rank")==2
)
result.show()

Convert order_date into a date type.



Create a window partitioned by customer_id.



Order by order_date ascending.



Use sum() with .over() to calculate running_total.



Store the final DataFrame in result and display it.

In [0]:
from pyspark.sql .functions import *
from pyspark.sql.window import Window
df1 = df.withColumn(
    "order_date",
    to_date("order_date")
)
window_Spect = Window.partitionBy(
    "customer_id"
).orderBy(
        col("order_date").asc()
    ).rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
df2 = df1.withColumn(
    "running_total",
    sum('amount').over(window_Spect)
)
result = df2
result.show()


